In [ ]:
!pip install dm-haiku

# Install (if needed)
# pip install --quiet jax jaxlib optax dm-haiku

import math
import functools
import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk
import optax

jax.config.update("jax_enable_x64", True)

import itertools
from functools import partial

import matplotlib.pyplot as plt

Array = jnp.ndarray

print(jnp.array([0.]).dtype)   # should print float64

# @title Neural SDE surrogate — Example 5 (Section 6) — NO PRIORS (drift+diffusion), alternating training (IMPROVED)
# Learns f(x,y) and diffusion Σ(x,y) from increments, without using the analytic forms.
#
# True data generator (kept separate): dx = (a1/x) dt + dW1, dy = a2 dt + dW2
# Training model: ΔX ~ Normal( f(x,y) dt,  Σ(x,y) dt ), with Σ = L L^T (Cholesky, PSD)
#
# Improvements added (still "no priors"):
#   A) Replace finite-difference diffusion smoothness with exact JVP (Jacobian-vector product) penalty
#   B) Add whitening / calibration penalty: z = L_dt^{-1}(ΔX - f dt) should be ~ N(0,I)
#   C) Remove weight decay (often biases σ downward); keep optional tiny weight decay as 0 by default
#   D) Multiple drift steps per diffusion step + slightly smaller diffusion LR
#
# NOTE: This remains increment-likelihood training; no analytic drift/diffusion forms are used.

import math
from dataclasses import dataclass

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax

jax.config.update("jax_enable_x64", True)

# -------------------------------------------------------------------
# 0) Config
# -------------------------------------------------------------------
@dataclass
class CFG:
    # --- data ---
    T: float = 2.0
    dt: float = 0.01
    n_traj: int = 2048

    # initial distribution (keep x away from 0 for the generator only)
    x0_low: float = 0.5
    x0_high: float = 3.0
    y0_low: float = -2.0
    y0_high: float = 2.0
    x_eps: float = 0.2  # generator safety floor for |x|

    # --- model / training ---
    hidden: int = 128
    depth: int = 3

    steps: int = 20000
    batch_size: int = 8192

    # learning rates
    lr_f: float = 2e-3      # drift LR
    lr_L: float = 1e-3      # diffusion LR (slightly smaller)

    # schedule: more drift steps than diffusion steps per outer iter
    drift_steps_per_iter: int = 3
    diff_steps_per_iter: int = 1

    # regularization + optimization
    weight_decay: float = 0.0   # IMPORTANT: set to 0 to avoid σ bias (was 1e-6)
    grad_clip: float = 1.0

    # diffusion regularization (generic, not physics):
    smooth_weight: float = 1e-3   # JVP-based smoothness on L(x)
    var_weight: float = 5e-4      # stabilize diffusion scale within minibatch

    # NEW: whitening / calibration penalty
    whiten_weight: float = 5e-3   # try in [1e-3, 1e-2]

    # numerical floors
    sigma_floor: float = 1e-3

cfg = CFG()
key_main = jax.random.PRNGKey(0)

# -------------------------------------------------------------------
# 1) TRUE DATA GENERATOR (ground truth used ONLY here + for later comparison plots)
# -------------------------------------------------------------------
truth = dict(a1=1.0, a2=0.5, sigma=1.0)

def make_ex5_data(key, cfg: CFG, truth):
    a1, a2, sig, dt = truth["a1"], truth["a2"], truth["sigma"], cfg.dt
    N = int(cfg.T / dt)
    t = jnp.linspace(0.0, cfg.T, N + 1)

    kx0, ky0, kn = jax.random.split(key, 3)  # IMPORTANT: split keys
    x0 = jax.random.uniform(kx0, (cfg.n_traj,), minval=cfg.x0_low, maxval=cfg.x0_high, dtype=jnp.float64)
    y0 = jax.random.uniform(ky0, (cfg.n_traj,), minval=cfg.y0_low, maxval=cfg.y0_high, dtype=jnp.float64)

    dW = jax.random.normal(kn, (cfg.n_traj, N, 2), dtype=jnp.float64) * math.sqrt(dt) * sig

    def step(state, dWn):
        x, y = state[:, 0], state[:, 1]
        x_safe = jnp.sign(x) * jnp.maximum(jnp.abs(x), cfg.x_eps)

        fx = a1 / x_safe
        fy = a2

        x1 = x + fx * dt + dWn[:, 0]
        y1 = y + fy * dt + dWn[:, 1]

        # keep away from 0 after update too (generator only)
        x1 = jnp.sign(x1) * jnp.maximum(jnp.abs(x1), cfg.x_eps)
        state1 = jnp.stack([x1, y1], axis=-1)
        return state1, state1

    state0 = jnp.stack([x0, y0], axis=-1)
    _, states = jax.lax.scan(step, state0, jnp.swapaxes(dW, 0, 1))  # (N, n_traj, 2)
    XY = jnp.concatenate([state0[None, :, :], states], axis=0)       # (N+1, n_traj, 2)
    XY = jnp.swapaxes(XY, 0, 1)                                      # (n_traj, N+1, 2)
    return t, XY

t, XY = make_ex5_data(key_main, cfg, truth)
print("Data shapes: t =", t.shape, ", XY =", XY.shape)

# quick plot x(t) for a few trajectories
t_np = np.asarray(t)
XY_np = np.asarray(XY)
plt.figure(figsize=(7,4))
for i in range(20):
    plt.plot(t_np[::2], XY_np[i, ::2, 0], alpha=0.5)
plt.xlabel("t"); plt.ylabel("x(t)")
plt.title("Example 5: sample x-trajectories (data generator)")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# -------------------------------------------------------------------
# 2) Build increment dataset: input [x,y] -> output Δ[x,y]
# -------------------------------------------------------------------
def build_increment_dataset(XY):
    XY_n   = XY[:, :-1, :]           # (n_traj, N, 2)
    XY_np1 = XY[:, 1:, :]            # (n_traj, N, 2)
    dXY    = XY_np1 - XY_n           # (n_traj, N, 2)

    X_in   = XY_n.reshape(-1, 2)     # (n_traj*N, 2)
    dX_out = dXY.reshape(-1, 2)      # (n_traj*N, 2)
    return X_in, dX_out

X_raw, dX = build_increment_dataset(XY)
print("\nIncrement dataset shapes: X_raw =", X_raw.shape, ", dX =", dX.shape)

#checks
dx_direct = XY[:, 1:, :] - XY[:, :-1, :]
dX_mat = dX.reshape(XY.shape[0], -1, 2)
print("\n[Validation CHECKS]")
print("dt from grid:", float(t[1]-t[0]), " cfg.dt:", cfg.dt)
print("max|Δ - (next-prev)| =", float(jnp.max(jnp.abs(dX_mat - dx_direct))))

# support check near y=0
y_all = np.asarray(X_raw[:, 1])
frac_y0 = float(np.mean(np.abs(y_all) < 0.05))
print(f"\n[SUPPORT CHECK] fraction of samples with |y|<0.05: {frac_y0:.4f}")
if frac_y0 < 0.01:
    print("WARNING: y=0 slice is mostly extrapolation. Consider plotting at y=median(y) instead.")

# -------------------------------------------------------------------
# 3) Normalize inputs
# -------------------------------------------------------------------
class Normalizer:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std
    def __call__(self, x):
        return (x - self.mean) / (self.std + 1e-8)

def fit_normalizer(X):
    return Normalizer(jnp.mean(X, axis=0), jnp.std(X, axis=0))

in_norm = fit_normalizer(X_raw)
X = in_norm(X_raw)

print("\n[NORMALIZATION CHECK]")
print("mean ~", np.asarray(jax.device_get(X.mean(0))), " std ~", np.asarray(jax.device_get(X.std(0))))

# -------------------------------------------------------------------
# 4) Model: drift MLP + diffusion-Cholesky MLP (PSD covariance)
# -------------------------------------------------------------------
def glorot(k, fan_in, fan_out):
    lim = math.sqrt(6.0 / (fan_in + fan_out))
    return jax.random.uniform(k, (fan_in, fan_out), minval=-lim, maxval=lim, dtype=jnp.float64)

def init_mlp(key, in_dim, out_dim, hidden, depth):
    keys = jax.random.split(key, depth+1)
    dims = [in_dim] + [hidden]*depth + [out_dim]
    params = []
    for i in range(len(dims)-1):
        params.append({
            "W": glorot(keys[i], dims[i], dims[i+1]),
            "b": jnp.zeros((dims[i+1],), dtype=jnp.float64),
        })
    return params

def mlp(params, x):
    h = x
    for i, layer in enumerate(params):
        h = h @ layer["W"] + layer["b"]
        if i < len(params)-1:
            h = jax.nn.swish(h)
    return h

def init_model(key, cfg: CFG):
    kf, kL = jax.random.split(key, 2)
    pf = init_mlp(kf, in_dim=2, out_dim=2, hidden=cfg.hidden, depth=cfg.depth)
    pL = init_mlp(kL, in_dim=2, out_dim=3, hidden=cfg.hidden, depth=cfg.depth)
    return {"pf": pf, "pL": pL}

def drift_and_chol(params, x_norm, cfg: CFG):
    f = mlp(params["pf"], x_norm)  # (...,2)

    raw = mlp(params["pL"], x_norm)  # (...,3)
    l11_raw = raw[..., 0]
    l21     = raw[..., 1]
    l22_raw = raw[..., 2]

    l11 = jax.nn.softplus(l11_raw) + cfg.sigma_floor
    l22 = jax.nn.softplus(l22_raw) + cfg.sigma_floor

    zeros = jnp.zeros_like(l11)
    L = jnp.stack([
        jnp.stack([l11, zeros], axis=-1),
        jnp.stack([l21, l22], axis=-1),
    ], axis=-2)
    return f, L  # Σ = L L^T

# -------------------------------------------------------------------
# 5) Increment NLL loss (multivariate Gaussian), + regularizers (JVP smoothness + whitening)
# -------------------------------------------------------------------
def l2_tree(p):
    return sum([jnp.sum(v**2) for v in jax.tree_util.tree_leaves(p)])

def mvn_nll(dX, mean, L_dt):
    r = dX - mean
    z = jax.vmap(lambda Li, ri: jax.scipy.linalg.solve_triangular(Li, ri, lower=True))(L_dt, r)
    quad = jnp.sum(z*z, axis=-1)
    logdet = 2.0 * jnp.log(jnp.clip(jnp.diagonal(L_dt, axis1=-2, axis2=-1), 1e-12, None)).sum(axis=-1)
    return 0.5 * (quad + logdet)

def diffusion_smoothness_penalty(params, xb_norm, cfg: CFG, key):
    # Exact JVP penalty: penalize ||J_L(x) v||^2 for random v
    v = jax.random.normal(key, xb_norm.shape, dtype=xb_norm.dtype)
    v = v / (jnp.linalg.norm(v, axis=-1, keepdims=True) + 1e-12)

    def L_fn(x):
        return drift_and_chol(params, x, cfg)[1]  # (...,2,2)

    _, dL = jax.jvp(L_fn, (xb_norm,), (v,))
    return jnp.mean(dL * dL)

def diffusion_variance_penalty(params, xb_norm, cfg: CFG):
    _, L = drift_and_chol(params, xb_norm, cfg)
    diag = jnp.diagonal(L, axis1=-2, axis2=-1)  # (B,2)
    return jnp.mean(jnp.var(diag, axis=0))

def whiten_penalty(dX, mean, L_dt):
    # z = L_dt^{-1} (dX - mean) should be ~ N(0, I)
    r = dX - mean
    z = jax.vmap(lambda Li, ri: jax.scipy.linalg.solve_triangular(Li, ri, lower=True))(L_dt, r)  # (B,2)

    z_mean = jnp.mean(z, axis=0)
    zc = z - z_mean
    B = z.shape[0]
    cov = (zc.T @ zc) / jnp.maximum(B - 1, 1)

    I = jnp.eye(2, dtype=z.dtype)
    return jnp.sum(z_mean**2) + jnp.sum((cov - I)**2)

def loss_total(params, xb_norm, dxb, cfg: CFG, key_smooth):
    f, L = drift_and_chol(params, xb_norm, cfg)
    mean = f * cfg.dt
    L_dt = L * math.sqrt(cfg.dt)

    nll = jnp.mean(mvn_nll(dxb, mean, L_dt))

    wd = cfg.weight_decay * l2_tree(params)  # default 0
    smooth = cfg.smooth_weight * diffusion_smoothness_penalty(params, xb_norm, cfg, key_smooth)
    varpen = cfg.var_weight * diffusion_variance_penalty(params, xb_norm, cfg)
    white = cfg.whiten_weight * whiten_penalty(dxb, mean, L_dt)

    return nll + wd + smooth + varpen + white, (nll, smooth, varpen, white)

# -------------------------------------------------------------------
# 6) Alternating training loop (multiple drift steps, then diffusion)
# -------------------------------------------------------------------
key_main, k_model = jax.random.split(key_main, 2)
params = init_model(k_model, cfg)

opt_f = optax.chain(optax.clip_by_global_norm(cfg.grad_clip),
                    optax.adamw(cfg.lr_f, weight_decay=0.0))
opt_L = optax.chain(optax.clip_by_global_norm(cfg.grad_clip),
                    optax.adamw(cfg.lr_L, weight_decay=0.0))

opt_state_f = opt_f.init(params["pf"])
opt_state_L = opt_L.init(params["pL"])

rng_np = np.random.default_rng(0)

def sample_minibatch(X_norm, dX, bs):
    N = X_norm.shape[0]
    idx = rng_np.choice(N, size=min(bs, N), replace=False)
    return X_norm[idx], dX[idx]

@jax.jit
def step_drift(pf, pL, opt_state_f, xb_norm, dxb, key_smooth):
    def _loss(pf_):
        p = {"pf": pf_, "pL": jax.lax.stop_gradient(pL)}
        val, aux = loss_total(p, xb_norm, dxb, cfg, key_smooth)
        return val, aux
    (val, aux), grads = jax.value_and_grad(_loss, has_aux=True)(pf)
    updates, opt_state_f2 = opt_f.update(grads, opt_state_f, pf)
    pf2 = optax.apply_updates(pf, updates)
    return pf2, opt_state_f2, val, aux

@jax.jit
def step_diffusion(pf, pL, opt_state_L, xb_norm, dxb, key_smooth):
    def _loss(pL_):
        p = {"pf": jax.lax.stop_gradient(pf), "pL": pL_}
        val, aux = loss_total(p, xb_norm, dxb, cfg, key_smooth)
        return val, aux
    (val, aux), grads = jax.value_and_grad(_loss, has_aux=True)(pL)
    updates, opt_state_L2 = opt_L.update(grads, opt_state_L, pL)
    pL2 = optax.apply_updates(pL, updates)
    return pL2, opt_state_L2, val, aux

print_every = 200
loss_hist = []

for step in range(1, cfg.steps + 1):
    xb, dxb = sample_minibatch(np.asarray(X), np.asarray(dX), cfg.batch_size)
    xb = jnp.asarray(xb)
    dxb = jnp.asarray(dxb)

    # multiple drift steps
    for _ in range(cfg.drift_steps_per_iter):
        key_main, ks = jax.random.split(key_main, 2)
        params["pf"], opt_state_f, val1, aux1 = step_drift(params["pf"], params["pL"], opt_state_f, xb, dxb, ks)

    # fewer diffusion steps
    for _ in range(cfg.diff_steps_per_iter):
        key_main, ks = jax.random.split(key_main, 2)
        params["pL"], opt_state_L, val2, aux2 = step_diffusion(params["pf"], params["pL"], opt_state_L, xb, dxb, ks)

    loss_hist.append(float(val2))

    if step % print_every == 0 or step == 1 or step == cfg.steps:
        nll, smooth, varpen, white = aux2
        print(
            f"step {step:5d}/{cfg.steps} | total = {float(val2):.6e} "
            f"| nll={float(nll):.6e} | smooth={float(smooth):.3e} | var={float(varpen):.3e} | white={float(white):.3e}"
        )

print("\nTraining finished.")

# -------------------------------------------------------------------
# 7) Diagnostics + comparison (comparison uses truth ONLY here)
# -------------------------------------------------------------------
def eval_slice(params, x_vals, y0):
    y_vals = jnp.full_like(x_vals, y0)
    XY_raw = jnp.stack([x_vals, y_vals], axis=-1)
    XY_norm = in_norm(XY_raw)
    f_hat, L_hat = drift_and_chol(params, XY_norm, cfg)
    Sigma = jnp.einsum("...ik,...jk->...ij", L_hat, L_hat)  # L L^T
    sigx = jnp.sqrt(jnp.clip(Sigma[...,0,0], 1e-12, None))
    sigy = jnp.sqrt(jnp.clip(Sigma[...,1,1], 1e-12, None))
    return XY_raw, f_hat, sigx, sigy

y0 = float(np.median(np.asarray(X_raw[:,1])))
print(f"\nUsing slice y0 = median(y) = {y0:.3f}  (set y0=0.0 if enough support is there)")

x_vals = jnp.linspace(0.5, 2.0, 200)
XY_s, f_s, sigx_s, sigy_s = eval_slice(params, x_vals, y0=y0)

x_np = np.asarray(XY_s[:,0])
fx_hat = np.asarray(f_s[:,0])
fy_hat = np.asarray(f_s[:,1])
sx_hat = np.asarray(sigx_s)
sy_hat = np.asarray(sigy_s)

fx_true = truth["a1"] / x_np
fy_true = truth["a2"] * np.ones_like(x_np)
sig_true = truth["sigma"] * np.ones_like(x_np)

plt.figure(figsize=(7,4))
plt.plot(x_np, fx_hat, lw=2, label=r"$\hat f_x$")
plt.plot(x_np, fx_true, lw=2, label=r"true $a_1/x$")
plt.xlabel("x"); plt.ylabel("drift in x")
plt.title(f"Drift-x slice at y={y0:.2f} (comparison only)")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(7,4))
plt.plot(x_np, fy_hat, lw=2, label=r"$\hat f_y$")
plt.plot(x_np, fy_true, lw=2, ls="--", label=r"true $a_2$")
plt.xlabel("x"); plt.ylabel("drift in y")
plt.title(f"Drift-y slice at y={y0:.2f} (comparison only)")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(7,4))
plt.plot(x_np, sx_hat, lw=2, label=r"$\hat\sigma_x$")
plt.plot(x_np, sy_hat, lw=2, label=r"$\hat\sigma_y$")
plt.plot(x_np, sig_true, lw=2, ls="--", label="true σ")
plt.xlabel("x"); plt.ylabel("diffusion (marginal σ)")
plt.title(f"Diffusion slice at y={y0:.2f} (comparison only)")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4))
plt.plot(np.arange(1, cfg.steps+1), loss_hist, lw=2)
plt.xlabel("step"); plt.ylabel("loss")
plt.title("Training loss (no-prior drift+diffusion) — improved")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

globals().update({"cfg_ex5": cfg, "params_ex5": params, "in_norm_ex5": in_norm})
print("\nExported: cfg_ex5, params_ex5, in_norm_ex5.")

# === Animations: drift and diffusion vs x as time evolves (2D SDE surrogate) ===
# FIX: set y-limits BEFORE animating (blit=True doesn't autoscale after empty init)

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# ----------------- bind names expected by "downstream" style -----------------
params_sde = params_ex5
in_norm_sde = in_norm_ex5
cfg = cfg_ex5

# ----------------- helpers ---------------------------------------------------
def _sigma_marginals_from_L(L_hat):
    Sigma = np.einsum("...ik,...jk->...ij", L_hat, L_hat)
    sigx = np.sqrt(np.clip(Sigma[..., 0, 0], 1e-12, None))
    sigy = np.sqrt(np.clip(Sigma[..., 1, 1], 1e-12, None))
    sigxy = Sigma[..., 0, 1]
    return sigx, sigy, sigxy

def eval_fx_fy_sigmas(params, in_norm, y0, x_min=0.25, x_max=2.5, n_points=200):
    x_vals = jnp.linspace(x_min, x_max, n_points)
    y_vals = jnp.full_like(x_vals, y0)
    XY_raw = jnp.stack([x_vals, y_vals], axis=-1)     # (N,2)
    XY_norm = in_norm(XY_raw)

    f_hat, L_hat = drift_and_chol(params, XY_norm, cfg)  # f_hat (N,2), L_hat (N,2,2)

    x_np = np.asarray(x_vals)
    fx_np = np.asarray(f_hat[:, 0])
    fy_np = np.asarray(f_hat[:, 1])
    L_np = np.asarray(L_hat)

    sigx_np, sigy_np, sigxy_np = _sigma_marginals_from_L(L_np)
    return x_np, fx_np, fy_np, sigx_np, sigy_np, sigxy_np

# ----------------- choose animation frames from the simulated time series -----
if ("t" in globals()) and ("XY" in globals()):
    t_array = np.asarray(t)
    XY_np = np.asarray(XY)  # (n_traj, Nt, 2)
    Nt = t_array.shape[0]
    y_med = np.median(XY_np[:, :, 1], axis=0)  # (Nt,)

    n_frames = 100
    frame_ids = np.linspace(0, Nt - 1, n_frames).astype(int)
else:
    t_array = np.linspace(0.0, float(getattr(cfg, "T", 2.0)), 101)
    y_med = np.linspace(-1.0, 1.0, t_array.shape[0])
    n_frames = 100
    frame_ids = np.linspace(0, len(t_array) - 1, n_frames).astype(int)

# x-range for plots (avoid x=0 singularity)
x_min, x_max = 0.25, 2.5

# Optional truth overlays
have_truth = "truth" in globals()
a1_true = float(truth["a1"]) if have_truth and ("a1" in truth) else None
a2_true = float(truth["a2"]) if have_truth and ("a2" in truth) else None
sig_true = float(truth["sigma"]) if have_truth and ("sigma" in truth) else 1.0

# ----------------- PRECOMPUTE Y-LIMS (critical fix for blit=True) ------------
probe_k = min(10, len(frame_ids))
probe_ids = np.linspace(0, len(frame_ids) - 1, probe_k).astype(int)

fx_all = []
sx_all = []
sy_all = []

for kk in probe_ids:
    idx = frame_ids[kk]
    y0 = float(y_med[idx])
    x_vals, fx_hat, _, sigx_hat, sigy_hat, _ = eval_fx_fy_sigmas(
        params_sde, in_norm_sde, y0, x_min=x_min, x_max=x_max
    )
    fx_all.append(fx_hat)
    sx_all.append(sigx_hat)
    sy_all.append(sigy_hat)

fx_all = np.concatenate(fx_all, axis=0)
sx_all = np.concatenate(sx_all, axis=0)
sy_all = np.concatenate(sy_all, axis=0)

# ignore NaNs/Infs if any
fx_finite = fx_all[np.isfinite(fx_all)]
sx_finite = sx_all[np.isfinite(sx_all)]
sy_finite = sy_all[np.isfinite(sy_all)]

# fallback if everything is non-finite (shouldn't happen, but keeps plot from crashing)
if fx_finite.size == 0:
    fx_lo, fx_hi = -1.0, 1.0
else:
    fx_lo, fx_hi = np.nanmin(fx_finite), np.nanmax(fx_finite)

if sx_finite.size == 0 or sy_finite.size == 0:
    s_lo, s_hi = 0.0, 2.0
else:
    s_lo = min(np.nanmin(sx_finite), np.nanmin(sy_finite))
    s_hi = max(np.nanmax(sx_finite), np.nanmax(sy_finite))

# add padding
fx_pad = 0.10 * (fx_hi - fx_lo + 1e-9)
s_pad  = 0.10 * (s_hi - s_lo + 1e-9)

fx_ylim = (fx_lo - fx_pad, fx_hi + fx_pad)
s_ylim  = (s_lo - s_pad,  s_hi + s_pad)

# ================== 1) Drift-x animation =====================================
fig_drift, ax_drift = plt.subplots(figsize=(6, 4))
line_fx, = ax_drift.plot([], [], lw=2, label=r"learned $\hat f_x(x,y)$")

if a1_true is not None:
    gt_fx_line, = ax_drift.plot([], [], lw=2, ls="--", label=r"true $a_1/x$")

ax_drift.set_xlim(x_min, x_max)
ax_drift.set_ylim(*fx_ylim)  # <-- FIX
ax_drift.set_xlabel("x")
ax_drift.set_ylabel("drift-x")
title_drift = ax_drift.set_title("")
ax_drift.grid(True, alpha=0.3)
ax_drift.legend(loc="upper right")

def init_drift():
    line_fx.set_data([], [])
    if a1_true is not None:
        gt_fx_line.set_data([], [])
    title_drift.set_text("")
    return (line_fx, gt_fx_line, title_drift) if a1_true is not None else (line_fx, title_drift)

def update_drift(frame_k):
    idx = frame_ids[frame_k]
    t0 = float(t_array[idx])
    y0 = float(y_med[idx])

    x_vals, fx_hat, _, _, _, _ = eval_fx_fy_sigmas(params_sde, in_norm_sde, y0, x_min=x_min, x_max=x_max)
    line_fx.set_data(x_vals, fx_hat)

    if a1_true is not None:
        gt_fx_line.set_data(x_vals, a1_true / x_vals)

    title_drift.set_text(rf"$\hat f_x(x,y)$ slice at $y={y0:.3f}$ (frame $t\approx{t0:.2f}$)")
    return (line_fx, gt_fx_line, title_drift) if a1_true is not None else (line_fx, title_drift)

anim_drift = FuncAnimation(
    fig_drift,
    update_drift,
    init_func=init_drift,
    frames=len(frame_ids),
    interval=80,
    blit=True,
)
plt.close(fig_drift)
display(HTML(anim_drift.to_jshtml()))

# ================== 2) Diffusion marginals animation =========================
fig_diff, ax_diff = plt.subplots(figsize=(6, 4))
line_sx, = ax_diff.plot([], [], lw=2, label=r"learned $\hat\sigma_x(x,y)$")
line_sy, = ax_diff.plot([], [], lw=2, label=r"learned $\hat\sigma_y(x,y)$")
ax_diff.axhline(sig_true, color="k", linestyle="--", label=rf"true $\sigma={sig_true}$")

ax_diff.set_xlim(x_min, x_max)
ax_diff.set_ylim(*s_ylim)  # <-- FIX
ax_diff.set_xlabel("x")
ax_diff.set_ylabel("diffusion (marginal)")
title_diff = ax_diff.set_title("")
ax_diff.grid(True, alpha=0.3)
ax_diff.legend(loc="upper right")

def init_diff():
    line_sx.set_data([], [])
    line_sy.set_data([], [])
    title_diff.set_text("")
    return line_sx, line_sy, title_diff

def update_diff(frame_k):
    idx = frame_ids[frame_k]
    t0 = float(t_array[idx])
    y0 = float(y_med[idx])

    x_vals, _, _, sigx_hat, sigy_hat, _ = eval_fx_fy_sigmas(params_sde, in_norm_sde, y0, x_min=x_min, x_max=x_max)
    line_sx.set_data(x_vals, sigx_hat)
    line_sy.set_data(x_vals, sigy_hat)
    title_diff.set_text(rf"$\hat\sigma(x,y)$ marginals at $y={y0:.3f}$ (frame $t\approx{t0:.2f}$)")
    return line_sx, line_sy, title_diff

anim_diff = FuncAnimation(
    fig_diff,
    update_diff,
    init_func=init_diff,
    frames=len(frame_ids),
    interval=80,
    blit=True,
)
plt.close(fig_diff)
display(HTML(anim_diff.to_jshtml()))

In [ ]:
# ============================================================
# EX5 (2D) — Generator Nets + Losses S1–S7 + Training + Span Check
# (COPY/PASTE THIS WHOLE CELL)
#
# This cell is robust:
#   - Removes the need for any "assert" placeholder lines.
#   - Auto-creates TX_gen from (t, XY) if missing.
#   - Auto-defines mu_fn/sig_fn from drift_and_chol + params_ex5 + in_norm_ex5 if not already defined.
#   - Defines surrogate_f_sigma in the style some downstream code expects.
# ============================================================

import math
from dataclasses import dataclass
import numpy as np
import jax
import jax.numpy as jnp
import optax

jax.config.update("jax_enable_x64", True)

# ---------------------------
# 0) Normalizer tools
# ---------------------------
class Normalizer:
    def __init__(self, mean, std, eps=1e-8):
        self.mean = jnp.asarray(mean, dtype=jnp.float64)
        self.std  = jnp.asarray(std,  dtype=jnp.float64)
        self.eps  = float(eps)
    def __call__(self, x):
        x = jnp.asarray(x, dtype=jnp.float64)
        return (x - self.mean) / (self.std + self.eps)

def fit_normalizer(X):
    X = jnp.asarray(X, dtype=jnp.float64)
    return Normalizer(jnp.mean(X, axis=0, keepdims=True), jnp.std(X, axis=0, keepdims=True))

# ---------------------------
# 1) Simple MLP helpers (generator nets)
# ---------------------------
def init_mlp_params(key, sizes, scale=1e-1, dtype=jnp.float64):
    keys = jax.random.split(key, len(sizes) - 1)
    params = []
    for k, (din, dout) in zip(keys, zip(sizes[:-1], sizes[1:])):
        W = scale * jax.random.normal(k, (din, dout), dtype=dtype)
        b = jnp.zeros((dout,), dtype=dtype)
        params.append((W, b))
    return params

def mlp_forward(params, x, activation="tanh"):
    h = x
    for (W, b) in params[:-1]:
        h = h @ W + b
        if activation == "tanh":
            h = jnp.tanh(h)
        elif activation == "relu":
            h = jnp.maximum(h, 0)
        elif activation == "gelu":
            h = 0.5 * h * (1.0 + jax.lax.erf(h / jnp.sqrt(2.0)))
        else:
            raise ValueError(f"Unknown activation: {activation}")
    W, b = params[-1]
    return h @ W + b

# ---------------------------
# 2) Ensure key_main exists
# ---------------------------
key_main = globals().get("key_main", jax.random.PRNGKey(0))

# ---------------------------
# 3) Ensure cfg_ex5/params_ex5/in_norm_ex5 exist (trained surrogate)
# ---------------------------
if "cfg_ex5" not in globals() or "params_ex5" not in globals() or "in_norm_ex5" not in globals():
    raise NameError(
        "Missing cfg_ex5 / params_ex5 / in_norm_ex5.\n"
        "Run cell first (the drift+chol MLP training)."
    )

cfg_ex5 = globals()["cfg_ex5"]
params_ex5 = globals()["params_ex5"]
in_norm_ex5 = globals()["in_norm_ex5"]

# surrogate forward
if "drift_and_chol" not in globals():
    raise NameError("Missing drift_and_chol(params, x_norm, cfg).")

drift_and_chol = globals()["drift_and_chol"]

# ---------------------------
# 4) Ensure TX_gen exists (t,x,y triples for sampling)
# ---------------------------
TX_gen = globals().get("TX_gen", None)

def _build_TX_gen_from_sim():
    if ("t" in globals()) and ("XY" in globals()):
        t_arr = jnp.asarray(globals()["t"], dtype=jnp.float64)                 # (Nt,)
        XY    = jnp.asarray(globals()["XY"], dtype=jnp.float64)                # (n_traj, Nt, 2)
        n_traj, Nt, _ = XY.shape
        t_tile = jnp.tile(t_arr[None, :], (n_traj, 1))                         # (n_traj, Nt)
        TX = jnp.stack([t_tile, XY[..., 0], XY[..., 1]], axis=-1)              # (n_traj, Nt, 3)
        TX = TX.reshape(-1, 3)                                                 # (n_traj*Nt, 3)
        return TX
    return None

if TX_gen is None:
    TX_gen = _build_TX_gen_from_sim()
    if TX_gen is None:
        print("[warn] TX_gen not found and (t,XY) not found. Will use uniform-box sampler only.")
    else:
        print("[ok] Built TX_gen from (t, XY):", TX_gen.shape)
    globals()["TX_gen"] = TX_gen

# ---------------------------
# 5) Build normalizers for generator nets: t_norm_gen, tx_norm_gen, txy_norm_gen
# ---------------------------
def _ensure_gen_normalizers(TX_gen):
    global t_norm_gen, tx_norm_gen, txy_norm_gen

    if callable(globals().get("t_norm_gen", None)) and callable(globals().get("tx_norm_gen", None)):
        t_norm_gen = globals()["t_norm_gen"]
        tx_norm_gen = globals()["tx_norm_gen"]
        txy_norm_gen = globals().get("txy_norm_gen", tx_norm_gen)
        globals()["txy_norm_gen"] = txy_norm_gen
        return

    if TX_gen is None:
        def _id_t(Z):  return jnp.asarray(Z, dtype=jnp.float64)
        def _id_tx(Z): return jnp.asarray(Z, dtype=jnp.float64)
        t_norm_gen = _id_t
        tx_norm_gen = _id_tx
        txy_norm_gen = _id_tx
        globals()["t_norm_gen"] = t_norm_gen
        globals()["tx_norm_gen"] = tx_norm_gen
        globals()["txy_norm_gen"] = txy_norm_gen
        print("[warn] Using identity generator normalizers (no TX_gen).")
        return

    TX_np = np.asarray(jax.device_get(jnp.asarray(TX_gen, dtype=jnp.float64)))
    TX_np = TX_np[np.isfinite(TX_np).all(axis=1)]
    if TX_np.shape[0] == 0:
        def _id_t(Z):  return jnp.asarray(Z, dtype=jnp.float64)
        def _id_tx(Z): return jnp.asarray(Z, dtype=jnp.float64)
        t_norm_gen = _id_t
        tx_norm_gen = _id_tx
        txy_norm_gen = _id_tx
        globals()["t_norm_gen"] = t_norm_gen
        globals()["tx_norm_gen"] = tx_norm_gen
        globals()["txy_norm_gen"] = txy_norm_gen
        print("[warn] TX_gen had no finite rows; using identity normalizers.")
        return

    t_np   = TX_np[:, [0]]          # (N,1)
    txy_np = TX_np[:, :3]           # (N,3)

    t_norm_gen  = fit_normalizer(t_np)
    tx_norm_gen = fit_normalizer(txy_np)
    txy_norm_gen = tx_norm_gen      # IMPORTANT alias for S6

    globals()["t_norm_gen"] = t_norm_gen
    globals()["tx_norm_gen"] = tx_norm_gen
    globals()["txy_norm_gen"] = txy_norm_gen

    try:
        print("Gen t_norm mean/std:", np.asarray(jax.device_get(t_norm_gen.mean)).ravel(),
              np.asarray(jax.device_get(t_norm_gen.std)).ravel())
        print("Gen tx_norm mean/std:", np.asarray(jax.device_get(tx_norm_gen.mean)).ravel(),
              np.asarray(jax.device_get(tx_norm_gen.std)).ravel())
    except Exception:
        pass

_ensure_gen_normalizers(TX_gen)

# ---------------------------
# 6) Auto-wire mu_fn, sig_fn from the trained surrogate (NO need for surrogate_f_sigma)
# ---------------------------
# surrogate is time-homogeneous: drift_and_chol expects x_norm=(x,y) only; t is ignored.
def mu_fn(t, x, y):
    x = jnp.asarray(x, dtype=jnp.float64)
    y = jnp.asarray(y, dtype=jnp.float64)
    XY_raw = jnp.stack([x, y], axis=-1)                  # (...,2)
    XY_norm = in_norm_ex5(XY_raw)                        # (...,2)
    f_hat, _L = drift_and_chol(params_ex5, XY_norm, cfg_ex5)
    return jnp.asarray(f_hat, dtype=jnp.float64)         # (...,2)

def sig_fn(t, x, y):
    x = jnp.asarray(x, dtype=jnp.float64)
    y = jnp.asarray(y, dtype=jnp.float64)
    XY_raw = jnp.stack([x, y], axis=-1)                  # (...,2)
    XY_norm = in_norm_ex5(XY_raw)                        # (...,2)
    _f_hat, L = drift_and_chol(params_ex5, XY_norm, cfg_ex5)
    return jnp.asarray(L, dtype=jnp.float64)             # (...,2,2) used as σ

globals()["mu_fn"] = mu_fn
globals()["sig_fn"] = sig_fn

# Optional compatibility wrapper some older code expects:
def surrogate_f_sigma(params_sde, in_norm_sde, t, x, y, return_L=False):
    # params_sde/in_norm_sde are ignored here; we use params_ex5/in_norm_ex5 by design.
    f = mu_fn(t, x, y)
    L = sig_fn(t, x, y)
    Sigma = jnp.einsum("...ik,...jk->...ij", L, L)
    if return_L:
        return f, Sigma, L
    return f, Sigma

globals()["surrogate_f_sigma"] = surrogate_f_sigma
globals()["params_sde"] = params_ex5
globals()["in_norm_sde"] = in_norm_ex5

# ---------------------------
# 7) Generator network config + init
# ---------------------------
@dataclass
class GenConfig:
    n_generators: int = 3
    hidden_tau: int = 32
    hidden_xi: int = 64
    hidden_beta: int = 64
    activation: str = "tanh"

gen_cfg = globals().get("gen_cfg", GenConfig())
gen_cfg = GenConfig(**{**gen_cfg.__dict__, "n_generators": int(getattr(gen_cfg, "n_generators", 3))})
m = int(gen_cfg.n_generators)

def tau_forward(params, t_norm, activation="tanh"):
    return mlp_forward(params, t_norm, activation=activation)[..., 0:1]

def xi_forward(params, txy_norm, activation="tanh"):
    return mlp_forward(params, txy_norm, activation=activation)[..., 0:2]   # xi=(xi_x,xi_y)

def beta_forward(params, txy_norm, activation="tanh"):
    return mlp_forward(params, txy_norm, activation=activation)[..., 0:1]

def init_generator_params(key, gen_cfg: GenConfig):
    keys = jax.random.split(key, 3 * gen_cfg.n_generators)
    params_tau, params_xi, params_beta = [], [], []
    for i in range(gen_cfg.n_generators):
        k_tau, k_xi, k_beta = keys[3*i], keys[3*i+1], keys[3*i+2]
        params_tau.append(init_mlp_params(k_tau,  [1, gen_cfg.hidden_tau,  gen_cfg.hidden_tau,  1]))
        params_xi.append(init_mlp_params(k_xi,   [3, gen_cfg.hidden_xi,   gen_cfg.hidden_xi,   2]))
        params_beta.append(init_mlp_params(k_beta,[3, gen_cfg.hidden_beta, gen_cfg.hidden_beta, 1]))
    return {"tau": params_tau, "xi": params_xi, "beta": params_beta}

params_gen = globals().get("params_gen", None)
if params_gen is None:
    key_main, key_gen = jax.random.split(key_main, 2)
    params_gen = init_generator_params(key_gen, gen_cfg)
    globals()["params_gen"] = params_gen

# ---------------------------
# 8) Evaluators
# ---------------------------
def eval_generators(params_gen, t, x, y=None, u=None, activation=None, return_phi=False):
    if activation is None:
        activation = gen_cfg.activation
    t_arr = jnp.asarray(t, dtype=jnp.float64)
    x_arr = jnp.asarray(x, dtype=jnp.float64)
    y_arr = jnp.zeros_like(x_arr) if (y is None) else jnp.asarray(y, dtype=jnp.float64)
    t_b, x_b, y_b = jnp.broadcast_arrays(t_arr, x_arr, y_arr)
    B_shape = t_b.shape

    t_flat  = t_b.reshape(-1, 1)
    x_flat  = x_b.reshape(-1, 1)
    y_flat  = y_b.reshape(-1, 1)
    txy_flat = jnp.concatenate([t_flat, x_flat, y_flat], axis=1)  # (B,3)

    t_norm   = t_norm_gen(t_flat)
    txy_norm = tx_norm_gen(txy_flat)

    tau_list, xi_list, beta_list = [], [], []
    for p_tau, p_xi, p_beta in zip(params_gen["tau"], params_gen["xi"], params_gen["beta"]):
        tau_flat = tau_forward(p_tau, t_norm, activation=activation)      # (B,1)
        xi_flat  = xi_forward(p_xi,  txy_norm, activation=activation)     # (B,2)
        be_flat  = beta_forward(p_beta, txy_norm, activation=activation)  # (B,1)
        tau_list.append(tau_flat.reshape(B_shape))
        xi_list.append(xi_flat.reshape(B_shape + (2,)))
        beta_list.append(be_flat.reshape(B_shape))

    tau_vals  = jnp.stack(tau_list, axis=0)   # (m, ...)
    xi_vals   = jnp.stack(xi_list, axis=0)    # (m, ..., 2)
    beta_vals = jnp.stack(beta_list, axis=0)  # (m, ...)

    if return_phi:
        if u is None:
            raise ValueError("return_phi=True requires u.")
        u_arr = jnp.asarray(u, dtype=jnp.float64)
        u_b = jnp.broadcast_to(u_arr, B_shape)
        phi_vals = beta_vals * u_b[None, ...]
        return tau_vals, xi_vals, beta_vals, phi_vals
    return tau_vals, xi_vals, beta_vals

eval_generators_jit = jax.jit(eval_generators, static_argnames=("activation", "return_phi"))

def eval_generators_tau_xi(params_gen, t, x, y=None, *, activation="tanh", normalize_txy=None):
    t = jnp.asarray(t, dtype=jnp.float64)
    x = jnp.asarray(x, dtype=jnp.float64)
    y = jnp.zeros_like(x) if (y is None) else jnp.asarray(y, dtype=jnp.float64)
    if t.ndim != 1 or x.ndim != 1 or y.ndim != 1:
        raise ValueError("eval_generators_tau_xi expects 1D arrays (B,) for t,x,y")

    if normalize_txy is None:
        t_raw, x_raw, y_raw = t, x, y
    else:
        t_raw, x_raw, y_raw = normalize_txy(t, x, y)
        t_raw = jnp.asarray(t_raw, dtype=jnp.float64)
        x_raw = jnp.asarray(x_raw, dtype=jnp.float64)
        y_raw = jnp.asarray(y_raw, dtype=jnp.float64)

    t_col   = t_raw.reshape(-1, 1)
    x_col   = x_raw.reshape(-1, 1)
    y_col   = y_raw.reshape(-1, 1)
    txy_col = jnp.concatenate([t_col, x_col, y_col], axis=1)  # (B,3)

    tN   = t_norm_gen(t_col)
    txyN = tx_norm_gen(txy_col)

    taus, xis = [], []
    for p_tau, p_xi in zip(params_gen["tau"], params_gen["xi"]):
        tau = tau_forward(p_tau, tN, activation=activation).reshape(-1)  # (B,)
        xi  = xi_forward(p_xi,  txyN, activation=activation)             # (B,2)
        taus.append(tau)
        xis.append(xi)

    tau_all = jnp.stack(taus, axis=0)  # (m,B)
    xi_all  = jnp.stack(xis,  axis=0)  # (m,B,2)
    return tau_all, xi_all

eval_generators_tau_xi_jit = jax.jit(eval_generators_tau_xi, static_argnames=("activation", "normalize_txy"))

print(f"[OK] Generator nets ready: m={m}, xi is 2D, normalizers set (t_norm_gen, tx_norm_gen, txy_norm_gen).")

# ============================================================
# 9) Losses S1–S7 (2D-correct)
# ============================================================

def _ordered_pair_indices(n: int):
    idx = jnp.arange(n, dtype=jnp.int32)
    ii  = jnp.repeat(idx, repeats=n-1)
    base = jnp.arange(n - 1, dtype=jnp.int32)
    i_col = idx[:, None]
    jj_mat = base + (base >= i_col).astype(jnp.int32)
    jj = jj_mat.reshape(-1)
    return ii, jj

def make_s1_lie_loss(n_generators: int, rcond: float = 1e-6):
    idx_i, idx_j = _ordered_pair_indices(n_generators)
    reg = jnp.asarray(rcond, dtype=jnp.float64) ** 2

    def _tau_val_and_dt(params_tau_i, t_scalar):
        def tau_scalar(tt):
            t_arr = jnp.asarray([[tt]], dtype=jnp.float64)
            tN = t_norm_gen(t_arr)
            out = tau_forward(params_tau_i, tN, activation=gen_cfg.activation)
            return out[0, 0]
        return tau_scalar(t_scalar), jax.grad(tau_scalar)(t_scalar)

    def _xi_val_and_derivs(params_xi_i, t_scalar, x_scalar, y_scalar):
        def xi_vec(tt, xx, yy):
            txy = jnp.asarray([[tt, xx, yy]], dtype=jnp.float64)
            txyN = tx_norm_gen(txy)
            out = xi_forward(params_xi_i, txyN, activation=gen_cfg.activation)  # (1,2)
            return out[0]  # (2,)
        xi_val = xi_vec(t_scalar, x_scalar, y_scalar)
        xi_t = jnp.stack([jax.grad(lambda tt: xi_vec(tt, x_scalar, y_scalar)[c])(t_scalar) for c in (0,1)], axis=0)
        xi_x = jnp.stack([jax.grad(lambda xx: xi_vec(t_scalar, xx, y_scalar)[c])(x_scalar) for c in (0,1)], axis=0)
        xi_y = jnp.stack([jax.grad(lambda yy: xi_vec(t_scalar, x_scalar, yy)[c])(y_scalar) for c in (0,1)], axis=0)
        return xi_val, xi_t, xi_x, xi_y

    def _fields_and_derivs_at_point(params_gen, t_scalar, x_scalar, y_scalar):
        tau_list, tau_t_list = [], []
        xi_list, xi_t_list, xi_x_list, xi_y_list = [], [], [], []
        for p_tau, p_xi in zip(params_gen["tau"], params_gen["xi"]):
            tau_i, tau_t_i = _tau_val_and_dt(p_tau, t_scalar)
            xi_i, xi_t_i, xi_x_i, xi_y_i = _xi_val_and_derivs(p_xi, t_scalar, x_scalar, y_scalar)
            tau_list.append(tau_i); tau_t_list.append(tau_t_i)
            xi_list.append(xi_i);   xi_t_list.append(xi_t_i)
            xi_x_list.append(xi_x_i); xi_y_list.append(xi_y_i)
        tau   = jnp.stack(tau_list, axis=0)      # (m,)
        tau_t = jnp.stack(tau_t_list, axis=0)    # (m,)
        xi    = jnp.stack(xi_list, axis=0)       # (m,2)
        xi_t  = jnp.stack(xi_t_list, axis=0)     # (m,2)
        xi_x  = jnp.stack(xi_x_list, axis=0)     # (m,2)
        xi_y  = jnp.stack(xi_y_list, axis=0)     # (m,2)
        return tau, xi, tau_t, xi_t, xi_x, xi_y

    def _point_err_and_C(tau, xi, tau_t, xi_t, xi_x, xi_y):
        xi_xc, xi_yc = xi[:,0], xi[:,1]
        V = jnp.stack([tau, xi_xc, xi_yc], axis=0)  # (3,m)

        tau_i, tau_j = tau[idx_i], tau[idx_j]
        tau_t_i, tau_t_j = tau_t[idx_i], tau_t[idx_j]

        xi_i, xi_j = xi[idx_i], xi[idx_j]
        xi_t_i, xi_t_j = xi_t[idx_i], xi_t[idx_j]
        xi_x_i, xi_x_j = xi_x[idx_i], xi_x[idx_j]
        xi_y_i, xi_y_j = xi_y[idx_i], xi_y[idx_j]

        a = tau_i * tau_t_j - tau_j * tau_t_i

        b = (tau_i * xi_t_j[:,0] + xi_i[:,0]*xi_x_j[:,0] + xi_i[:,1]*xi_y_j[:,0]
             -tau_j * xi_t_i[:,0] - xi_j[:,0]*xi_x_i[:,0] - xi_j[:,1]*xi_y_i[:,0])

        c = (tau_i * xi_t_j[:,1] + xi_i[:,0]*xi_x_j[:,1] + xi_i[:,1]*xi_y_j[:,1]
             -tau_j * xi_t_i[:,1] - xi_j[:,0]*xi_x_i[:,1] - xi_j[:,1]*xi_y_i[:,1])

        B = jnp.stack([a,b,c], axis=0)  # (3,K)

        G = V @ V.T
        G_reg = G + reg * jnp.eye(3, dtype=G.dtype)
        X = jnp.linalg.solve(G_reg, B)   # (3,K)
        C = V.T @ X                      # (m,K)
        P_B = V @ C                      # (3,K)
        E = B - P_B
        err = jnp.sum(jnp.abs(E))
        return err, C

    def _loss_impl(params_gen, txy_batch):
        def eval_at_z(z):
            return _fields_and_derivs_at_point(params_gen, z[0], z[1], z[2])
        taus, xis, tau_ts, xi_ts, xi_xs, xi_ys = jax.vmap(eval_at_z)(txy_batch)
        errs, Cs = jax.vmap(_point_err_and_C)(taus, xis, tau_ts, xi_ts, xi_xs, xi_ys)
        error_sum = jnp.sum(errs)
        var_sum = jnp.sum(jnp.var(Cs, axis=0))
        return error_sum + var_sum, {"error_sum": error_sum, "var_sum": var_sum}

    return jax.jit(_loss_impl)

def make_s2_jacobi_loss_nested(n_generators: int):
    triples = [(i,j,k) for i in range(n_generators) for j in range(i+1,n_generators) for k in range(j+1,n_generators)]
    if not triples:
        def _zero(params_gen, txy_batch):
            return jnp.array(0.0, dtype=jnp.float64), {"per_point": jnp.zeros((txy_batch.shape[0],), dtype=jnp.float64), "num_triples": 0}
        return jax.jit(_zero)

    tri_i = jnp.array([t[0] for t in triples], dtype=jnp.int32)
    tri_j = jnp.array([t[1] for t in triples], dtype=jnp.int32)
    tri_k = jnp.array([t[2] for t in triples], dtype=jnp.int32)
    perms6 = jnp.array([[0,1,2],[0,2,1],[1,0,2],[1,2,0],[2,0,1],[2,1,0]], dtype=jnp.int32)

    def _fields_jac_hess(params_gen, z):
        F_list, J_list, H_list = [], [], []
        for p_tau, p_xi in zip(params_gen["tau"], params_gen["xi"]):
            def Xi(zz):
                tt, xx, yy = zz[0], zz[1], zz[2]
                t_arr  = jnp.asarray([[tt]], dtype=jnp.float64)
                txy_arr= jnp.asarray([[tt,xx,yy]], dtype=jnp.float64)
                tN  = t_norm_gen(t_arr)
                txyN= tx_norm_gen(txy_arr)
                tau = tau_forward(p_tau, tN, activation=gen_cfg.activation)[0,0]
                xi  = xi_forward(p_xi, txyN, activation=gen_cfg.activation)[0]  # (2,)
                return jnp.array([tau, xi[0], xi[1]], dtype=jnp.float64)
            Fi = Xi(z)
            Ji = jax.jacobian(Xi)(z)
            Hi = jax.jacobian(lambda zz: jax.jacobian(Xi)(zz))(z)
            F_list.append(Fi); J_list.append(Ji); H_list.append(Hi)
        return jnp.stack(F_list,0), jnp.stack(J_list,0), jnp.stack(H_list,0)

    def _bracket_val(F,J,p,q):
        return (J[q] @ F[p]) - (J[p] @ F[q])

    def _dir_along(F,J,H,r,p,q):
        Jr,Jp,Jq = J[r],J[p],J[q]
        Hp,Hq = H[p],H[q]
        fr,fp,fq = F[r],F[p],F[q]
        t1 = Jq @ (Jp @ fr)
        t2 = ((Hq * fr[None,None,:]).sum(axis=2)) @ fp
        t3 = Jp @ (Jq @ fr)
        t4 = ((Hp * fr[None,None,:]).sum(axis=2)) @ fq
        return t1 + t2 - t3 - t4

    def _double_bracket(F,J,H,r,p,q):
        inner = _bracket_val(F,J,p,q)
        return _dir_along(F,J,H,r,p,q) - (J[r] @ inner)

    def _jacobi_one_order(F,J,H,u,v,w):
        return _double_bracket(F,J,H,u,v,w) + _double_bracket(F,J,H,w,u,v) + _double_bracket(F,J,H,v,w,u)

    def _triple_sum_over_6(F,J,H,i,j,k):
        inds = jnp.array([i,j,k], dtype=jnp.int32)
        def _one_perm(p):
            u,v,w = inds[p[0]], inds[p[1]], inds[p[2]]
            r = _jacobi_one_order(F,J,H,u,v,w)
            return jnp.sum(jnp.abs(r))
        return jnp.sum(jax.vmap(_one_perm)(perms6))

    def _point_loss(params_gen, z):
        F,J,H = _fields_jac_hess(params_gen, z)
        per_tr = jax.vmap(lambda a,b,c: _triple_sum_over_6(F,J,H,a,b,c))(tri_i, tri_j, tri_k)
        return jnp.sum(per_tr)

    _pl = jax.jit(_point_loss)

    def _loss_impl(params_gen, txy_batch):
        per_point = jax.vmap(lambda z: _pl(params_gen, z))(txy_batch)
        return jnp.sum(per_point), {"per_point": per_point, "num_triples": int(tri_i.shape[0])}

    return jax.jit(_loss_impl)

def make_s3_skewsym_loss(n_generators: int):
    pairs = [(i,j) for i in range(n_generators) for j in range(i+1,n_generators)]
    if not pairs:
        def _zero(params_gen, txy_batch):
            return jnp.array(0.0, dtype=jnp.float64), {"per_point": jnp.zeros((txy_batch.shape[0],), dtype=jnp.float64), "num_pairs": 0}
        return jax.jit(_zero)

    pi = jnp.array([p[0] for p in pairs], dtype=jnp.int32)
    pj = jnp.array([p[1] for p in pairs], dtype=jnp.int32)

    def _fields_and_jac(params_gen, z):
        F_list, J_list = [], []
        for p_tau, p_xi in zip(params_gen["tau"], params_gen["xi"]):
            def Xi(zz):
                tt,xx,yy = zz[0], zz[1], zz[2]
                t_arr = jnp.asarray([[tt]], dtype=jnp.float64)
                txy_arr= jnp.asarray([[tt,xx,yy]], dtype=jnp.float64)
                tN  = t_norm_gen(t_arr)
                txyN= tx_norm_gen(txy_arr)
                tau = tau_forward(p_tau, tN, activation=gen_cfg.activation)[0,0]
                xi  = xi_forward(p_xi, txyN, activation=gen_cfg.activation)[0]
                return jnp.array([tau, xi[0], xi[1]], dtype=jnp.float64)
            Fi = Xi(z)
            Ji = jax.jacobian(Xi)(z)
            F_list.append(Fi); J_list.append(Ji)
        return jnp.stack(F_list,0), jnp.stack(J_list,0)

    def _bracket(F,J,p,q):
        return (J[q] @ F[p]) - (J[p] @ F[q])

    def _point_loss(params_gen, z):
        F,J = _fields_and_jac(params_gen, z)
        def one(i,j):
            r = _bracket(F,J,i,j) + _bracket(F,J,j,i)
            return jnp.sum(jnp.abs(r))
        return jnp.sum(jax.vmap(one)(pi,pj))

    _pl = jax.jit(_point_loss)

    def _loss_impl(params_gen, txy_batch):
        per_point = jax.vmap(lambda z: _pl(params_gen, z))(txy_batch)
        return jnp.sum(per_point), {"per_point": per_point, "num_pairs": int(pi.shape[0])}

    return jax.jit(_loss_impl)

def make_s4_bilinearity_loss(n_generators: int, num_cc: int = 4, cc_list=None, normalize: bool = True):
    triples = [(i,j,k) for i in range(n_generators) for j in range(i+1,n_generators) for k in range(j+1,n_generators)]
    if not triples:
        def _zero(params_gen, txy_batch, key=None):
            return jnp.array(0.0, dtype=jnp.float64), {"per_point": jnp.zeros((txy_batch.shape[0],), dtype=jnp.float64)}
        return jax.jit(_zero)

    tri_i = jnp.array([t[0] for t in triples], dtype=jnp.int32)
    tri_j = jnp.array([t[1] for t in triples], dtype=jnp.int32)
    tri_k = jnp.array([t[2] for t in triples], dtype=jnp.int32)
    perms6 = jnp.array([[0,1,2],[0,2,1],[1,0,2],[1,2,0],[2,0,1],[2,1,0]], dtype=jnp.int32)

    if cc_list is not None:
        cc_const = jnp.asarray(cc_list, dtype=jnp.float64)
    else:
        cc_const = jax.random.uniform(jax.random.PRNGKey(0), (num_cc,2), minval=-1.0, maxval=1.0, dtype=jnp.float64)

    def _fields_and_jac(params_gen, z):
        F_list, J_list = [], []
        for p_tau, p_xi in zip(params_gen["tau"], params_gen["xi"]):
            def Xi(zz):
                tt,xx,yy = zz[0], zz[1], zz[2]
                t_arr = jnp.asarray([[tt]], dtype=jnp.float64)
                txy_arr= jnp.asarray([[tt,xx,yy]], dtype=jnp.float64)
                tN = t_norm_gen(t_arr)
                txyN= tx_norm_gen(txy_arr)
                tau = tau_forward(p_tau, tN, activation=gen_cfg.activation)[0,0]
                xi  = xi_forward(p_xi, txyN, activation=gen_cfg.activation)[0]
                return jnp.array([tau, xi[0], xi[1]], dtype=jnp.float64)
            Fi = Xi(z)
            Ji = jax.jacobian(Xi)(z)
            F_list.append(Fi); J_list.append(Ji)
        return jnp.stack(F_list,0), jnp.stack(J_list,0)

    def _bracket(F,J,p,q):
        return (J[q] @ F[p]) - (J[p] @ F[q])

    def _triple_terms(F,J,i,j,k,cc):
        inds = jnp.array([i,j,k], dtype=jnp.int32)
        def one_perm(p):
            u,v,w = inds[p[0]], inds[p[1]], inds[p[2]]
            fu,fv,fw = F[u],F[v],F[w]
            Ju,Jv,Jw = J[u],J[v],J[w]

            def one_cc(cpair):
                c,cp = cpair[0], cpair[1]
                f_uv = c*fu + cp*fv
                J_uv = c*Ju + cp*Jv
                f_vw = c*fv + cp*fw
                J_vw = c*Jv + cp*Jw

                term1 = (Jw @ f_uv) - (J_uv @ fw)
                rhs1  = c*_bracket(F,J,u,w) + cp*_bracket(F,J,v,w)
                r1 = term1 - rhs1

                term2 = (J_vw @ fu) - (Ju @ f_vw)
                rhs2  = c*_bracket(F,J,u,v) + cp*_bracket(F,J,u,w)
                r2 = term2 - rhs2

                if normalize:
                    denom = jnp.abs(c) + jnp.abs(cp) + 1e-12
                    r1 = r1/denom; r2 = r2/denom
                return jnp.sum(jnp.abs(r1)) + jnp.sum(jnp.abs(r2))

            return jnp.mean(jax.vmap(one_cc)(cc))
        return jnp.sum(jax.vmap(one_perm)(perms6))

    def _point_loss(params_gen, z, cc):
        F,J = _fields_and_jac(params_gen, z)
        per_tr = jax.vmap(lambda a,b,c: _triple_terms(F,J,a,b,c,cc))(tri_i, tri_j, tri_k)
        return jnp.sum(per_tr)

    _pl = jax.jit(_point_loss)

    def _loss_impl(params_gen, txy_batch, key=None):
        cc = cc_const
        per_point = jax.vmap(lambda z: _pl(params_gen, z, cc))(txy_batch)
        return jnp.sum(per_point), {"per_point": per_point, "num_triples": int(tri_i.shape[0]), "num_cc": int(cc.shape[0])}

    return jax.jit(_loss_impl)

def make_s5_column_independence_loss(n_generators: int, *, mode: str = "sigma", tau: float = 0.0, eps: float = 1e-12):
    mode = "sigma" if mode == "sigma" else "corr_l2"
    mode_code = 0 if mode == "sigma" else 1

    def _A_from_batch(params_gen, txy_batch):
        tB = txy_batch[:,0]
        xB = txy_batch[:,1]
        yB = txy_batch[:,2]
        tau_vals, xi_vals, _ = eval_generators_jit(params_gen, tB, xB, yB)
        comp = jnp.concatenate([tau_vals[...,None], xi_vals], axis=2)  # (m,N,3)
        comp_N3m = jnp.transpose(comp, (1,2,0))                        # (N,3,m)
        return comp_N3m.reshape(-1, n_generators)                      # (3N,m)

    def _loss_impl(params_gen, txy_batch):
        A = _A_from_batch(params_gen, txy_batch)
        col_norms = jnp.linalg.norm(A, axis=0) + eps
        Ahat = A / col_norms
        G = Ahat.T @ Ahat
        if mode_code == 0:
            lam = jnp.linalg.eigvalsh(G)
            sigma_min = jnp.sqrt(jnp.clip(jnp.min(lam), 0.0, None))
            loss = jnp.maximum(0.0, jnp.asarray(tau, dtype=G.dtype) - sigma_min)
            return loss, {"sigma_min": sigma_min}
        else:
            I = jnp.eye(G.shape[0], dtype=G.dtype)
            off = G - I
            off = off - jnp.diag(jnp.diag(off))
            return jnp.sum(off*off), {"gram_diag_mean": jnp.mean(jnp.diag(G))}

    return jax.jit(_loss_impl)

def make_s6_commutator_loss_ito(*, mu_fn, sig_fn, use_abs: bool = False):
    def tau_val_and_dt(params_tau, t_scalar):
        def tau_scalar(tt):
            t_arr = jnp.asarray([[tt]], dtype=jnp.float64)
            tN = t_norm_gen(t_arr)
            out = tau_forward(params_tau, tN, activation=gen_cfg.activation)
            return out[0,0]
        return tau_scalar(t_scalar), jax.grad(tau_scalar)(t_scalar)

    def xi_val_jac_hess(params_xi, t_scalar, x_scalar, y_scalar):
        def Xi(z):
            tt,xx,yy = z[0],z[1],z[2]
            txy = jnp.asarray([[tt,xx,yy]], dtype=jnp.float64)
            txyN = txy_norm_gen(txy)
            out = xi_forward(params_xi, txyN, activation=gen_cfg.activation)
            return out[0,:]  # (2,)
        z0 = jnp.array([t_scalar,x_scalar,y_scalar], dtype=jnp.float64)
        xi = Xi(z0)
        Jz = jax.jacobian(Xi)(z0)                          # (2,3)
        Hz = jax.jacobian(lambda z: jax.jacobian(Xi)(z))(z0)  # (2,3,3)
        Hxy = Hz[:,1:,1:]                                 # (2,2,2)
        return xi, Jz, Hxy

    def f_sigma_and_derivs(t_scalar, x_scalar, y_scalar):
        def F(z):
            tt,xx,yy = z[0],z[1],z[2]
            return mu_fn(tt,xx,yy)  # (2,)
        z0 = jnp.array([t_scalar,x_scalar,y_scalar], dtype=jnp.float64)
        f = F(z0)
        Jz_f = jax.jacobian(F)(z0)                         # (2,3)
        f_t = Jz_f[:,0]
        Jf  = Jz_f[:,1:]                                   # (2,2)

        def S_flat(z):
            tt,xx,yy = z[0],z[1],z[2]
            S = jnp.asarray(sig_fn(tt,xx,yy), dtype=jnp.float64)  # (2,m) or (2,2)
            if S.ndim != 2 or S.shape[0] != 2:
                raise ValueError(f"sig_fn must return (2,m); got {S.shape}")
            return S.reshape(-1)
        s_flat = S_flat(z0)
        Jz_s = jax.jacobian(S_flat)(z0)                    # (2m,3)
        ds_dt  = Jz_s[:,0]
        ds_dxy = Jz_s[:,1:]                                # (2m,2)

        sigma = jnp.asarray(sig_fn(t_scalar,x_scalar,y_scalar), dtype=jnp.float64)
        mW = sigma.shape[1]
        sigma_t = ds_dt.reshape(2,mW)
        Jsigma_xy = ds_dxy.reshape(2,mW,2)                 # (2,m,2) last axis = (x,y)
        return f, f_t, Jf, sigma, sigma_t, Jsigma_xy

    def _point_residual(params_gen, z):
        t,x,y = z[0],z[1],z[2]
        f,f_t,Jf,sigma,sigma_t,Jsigma_xy = f_sigma_and_derivs(t,x,y)
        a = sigma @ sigma.T  # (2,2)
        total = jnp.array(0.0, dtype=jnp.float64)
        for p_tau, p_xi in zip(params_gen["tau"], params_gen["xi"]):
            tau_i, tau_t_i = tau_val_and_dt(p_tau, t)
            xi_i, Jz_xi, Hxy = xi_val_jac_hess(p_xi, t, x, y)
            xi_t = Jz_xi[:,0]
            Jxi  = Jz_xi[:,1:]  # (2,2)

            diff_vec = 0.5 * jnp.einsum("pq,rpq->r", a, Hxy)      # (2,)
            adv_xi   = Jxi @ f                                   # (2,)
            adv_f    = Jf  @ xi_i                                 # (2,)
            r1 = xi_t + adv_xi - adv_f - tau_i*f_t - f*tau_t_i + diff_vec

            Dxi_sigma = jnp.einsum("k,ikm->im", xi_i, Jsigma_xy)  # (2,m)
            r2 = (Jxi @ sigma) - Dxi_sigma - tau_i*sigma_t - 0.5*tau_t_i*sigma

            if use_abs:
                total = total + jnp.sum(jnp.abs(r1)) + jnp.sum(jnp.abs(r2))
            else:
                total = total + jnp.sum(r1*r1) + jnp.sum(r2*r2)
        return total

    def _loss_impl(params_gen, txy_batch):
        txy_batch = jnp.asarray(txy_batch, dtype=jnp.float64)
        per_point = jax.vmap(lambda z: _point_residual(params_gen, z))(txy_batch)
        return jnp.mean(per_point), {"per_point": per_point}

    return jax.jit(_loss_impl)

def make_s7_pushforward_coeff_loss_sde_only_2d(
    *, mu_fn, sig_fn, eps: float = 1e-2, num_steps: int = 1, sigma_floor: float = 1e-8,
    dt_neg_penalty: float = 100.0, activation: str = "tanh", normalize_txy=None, jit: bool = True, use_heun: bool = True,
    fd_t: float = 1e-3, fd_x: float = 1e-3, fd_y: float = 1e-3,
    tau_clip: float = 5.0, xi_clip: float = 5.0, xy_clip_abs: float = 50.0, x_min_abs: float = 1e-3,
):
    eval_gen_tau_xi = eval_generators_tau_xi_jit
    eps = jnp.asarray(eps, dtype=jnp.float64)
    num_steps = int(num_steps)
    fd_t = jnp.asarray(fd_t, dtype=jnp.float64)
    fd_x = jnp.asarray(fd_x, dtype=jnp.float64)
    fd_y = jnp.asarray(fd_y, dtype=jnp.float64)

    def _sanitize(t,x,y):
        t = jnp.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)
        x = jnp.clip(jnp.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0), -xy_clip_abs, xy_clip_abs)
        y = jnp.clip(jnp.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0), -xy_clip_abs, xy_clip_abs)
        x = jnp.sign(x) * jnp.maximum(jnp.abs(x), x_min_abs)
        return t,x,y

    def _mu_eval(t,x,y):
        out = jnp.asarray(mu_fn(t,x,y), dtype=jnp.float64)
        return jnp.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

    def _sig_eval(t,x,y):
        out = jnp.asarray(sig_fn(t,x,y), dtype=jnp.float64)
        out = jnp.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
        if out.shape[-2] != 2:
            raise ValueError(f"sig_fn must return (...,2,m); got {out.shape}")
        mW = out.shape[-1]
        return out, mW

    def _diag_from_flat(params_gen, t_flat, x_flat, y_flat):
        tau_all, xi_all = eval_gen_tau_xi(params_gen, t_flat, x_flat, y_flat, activation=activation, normalize_txy=normalize_txy)
        tau_all = jnp.asarray(tau_all, dtype=jnp.float64)
        xi_all  = jnp.asarray(xi_all,  dtype=jnp.float64)

        m = tau_all.shape[0]
        Bflat = tau_all.shape[1]
        if Bflat % m != 0:
            raise ValueError("Internal shape error: expected Bflat divisible by m.")
        B = Bflat // m

        tau_blk = tau_all.reshape(m, m, B)
        xi_blk  = xi_all.reshape(m, m, B, 2)
        idx = jnp.arange(m, dtype=jnp.int32)
        tau_d = tau_blk[idx, idx, :]
        xi_d  = xi_blk[idx, idx, :, :]

        tau_d = tau_clip * jnp.tanh(jnp.nan_to_num(tau_d)/tau_clip)
        xi_d  = xi_clip  * jnp.tanh(jnp.nan_to_num(xi_d)/xi_clip)
        return tau_d, xi_d

    def _rhs_diag_with_derivs(params_gen, tS, xS, yS):
        m,B = tS.shape
        t_flat = tS.reshape(-1); x_flat = xS.reshape(-1); y_flat = yS.reshape(-1)

        if normalize_txy is not None:
            t_flat, x_flat, y_flat = normalize_txy(t_flat, x_flat, y_flat)
            t_flat = jnp.asarray(t_flat, dtype=jnp.float64)
            x_flat = jnp.asarray(x_flat, dtype=jnp.float64)
            y_flat = jnp.asarray(y_flat, dtype=jnp.float64)

        tau0, xi0 = _diag_from_flat(params_gen, t_flat, x_flat, y_flat)

        tau_p, xi_p = _diag_from_flat(params_gen, t_flat+fd_t, x_flat, y_flat)
        tau_m, xi_m = _diag_from_flat(params_gen, t_flat-fd_t, x_flat, y_flat)
        tau_t = (tau_p - tau_m)/(2*fd_t)
        xi_t  = (xi_p  - xi_m )/(2*fd_t)

        _, xi_xp = _diag_from_flat(params_gen, t_flat, x_flat+fd_x, y_flat)
        _, xi_xm = _diag_from_flat(params_gen, t_flat, x_flat-fd_x, y_flat)
        xi_x  = (xi_xp - xi_xm)/(2*fd_x)
        xi_xx = (xi_xp - 2*xi0 + xi_xm)/(fd_x*fd_x)

        _, xi_yp = _diag_from_flat(params_gen, t_flat, x_flat, y_flat+fd_y)
        _, xi_ym = _diag_from_flat(params_gen, t_flat, x_flat, y_flat-fd_y)
        xi_y  = (xi_yp - xi_ym)/(2*fd_y)
        xi_yy = (xi_yp - 2*xi0 + xi_ym)/(fd_y*fd_y)

        _, xi_xp_yp = _diag_from_flat(params_gen, t_flat, x_flat+fd_x, y_flat+fd_y)
        _, xi_xp_ym = _diag_from_flat(params_gen, t_flat, x_flat+fd_x, y_flat-fd_y)
        _, xi_xm_yp = _diag_from_flat(params_gen, t_flat, x_flat-fd_x, y_flat+fd_y)
        _, xi_xm_ym = _diag_from_flat(params_gen, t_flat, x_flat-fd_x, y_flat-fd_y)
        xi_xy = (xi_xp_yp - xi_xp_ym - xi_xm_yp + xi_xm_ym)/(4*fd_x*fd_y)

        tau0  = tau0.reshape(m,B); tau_t = tau_t.reshape(m,B)
        xi0   = xi0.reshape(m,B,2); xi_t  = xi_t.reshape(m,B,2)
        xi_x  = xi_x.reshape(m,B,2); xi_y  = xi_y.reshape(m,B,2)
        xi_xx = xi_xx.reshape(m,B,2); xi_yy = xi_yy.reshape(m,B,2)
        xi_xy = xi_xy.reshape(m,B,2)
        return tau0, xi0, tau_t, xi_t, xi_x, xi_y, xi_xx, xi_yy, xi_xy

    def _a_from_sigma(sg):
        a = jnp.einsum("...iA,...jA->...ij", sg, sg)
        a = a + (sigma_floor**2)*jnp.eye(2, dtype=a.dtype)
        return a

    def _flow_step_rhs(params_gen, tS, xS, yS, muS, sgS):
        tau, xi, tau_t, xi_t, xi_x, xi_y, xi_xx, xi_yy, xi_xy = _rhs_diag_with_derivs(params_gen, tS, xS, yS)
        Jxi = jnp.stack([xi_x, xi_y], axis=-1)  # (m,B,2,2)
        a = _a_from_sigma(sgS)
        a11,a12,a22 = a[...,0,0], a[...,0,1], a[...,1,1]
        diff_vec = 0.5*(a11[...,None]*xi_xx + 2.0*a12[...,None]*xi_xy + a22[...,None]*xi_yy)
        adv_xi = jnp.einsum("...ij,...j->...i", Jxi, muS)
        k_t  = tau
        k_xy = xi
        k_mu = xi_t + adv_xi - muS*tau_t[...,None] + diff_vec
        k_sg = jnp.einsum("...ij,...jA->...iA", Jxi, sgS) - 0.5*tau_t[...,None,None]*sgS
        return k_t, k_xy, k_mu, k_sg

    def _flow_allgens(params_gen, t0, x0, y0):
        tau_all, _ = eval_gen_tau_xi(params_gen, t0, x0, y0, activation=activation, normalize_txy=normalize_txy)
        m = int(tau_all.shape[0]); B = int(t0.shape[0])

        tS = jnp.broadcast_to(t0[None,:], (m,B))
        xS = jnp.broadcast_to(x0[None,:], (m,B))
        yS = jnp.broadcast_to(y0[None,:], (m,B))

        t0c,x0c,y0c = _sanitize(t0,x0,y0)
        mu0 = _mu_eval(t0c,x0c,y0c)         # (B,2)
        sg0,mW = _sig_eval(t0c,x0c,y0c)     # (B,2,mW)

        muS = jnp.broadcast_to(mu0[None,:,:], (m,B,2))
        sgS = jnp.broadcast_to(sg0[None,:,:,:], (m,B,2,mW))

        def body(_, state):
            tS,xS,yS,muS,sgS = state
            if use_heun:
                k1_t,k1_xy,k1_mu,k1_sg = _flow_step_rhs(params_gen,tS,xS,yS,muS,sgS)
                tP = tS + eps*k1_t
                xP = xS + eps*k1_xy[...,0]
                yP = yS + eps*k1_xy[...,1]
                muP= muS+ eps*k1_mu
                sgP= sgS+ eps*k1_sg
                k2_t,k2_xy,k2_mu,k2_sg = _flow_step_rhs(params_gen,tP,xP,yP,muP,sgP)
                tN = tS + 0.5*eps*(k1_t + k2_t)
                xN = xS + 0.5*eps*(k1_xy[...,0]+k2_xy[...,0])
                yN = yS + 0.5*eps*(k1_xy[...,1]+k2_xy[...,1])
                muN= muS+ 0.5*eps*(k1_mu+k2_mu)
                sgN= sgS+ 0.5*eps*(k1_sg+k2_sg)
            else:
                k_t,k_xy,k_mu,k_sg = _flow_step_rhs(params_gen,tS,xS,yS,muS,sgS)
                tN = tS + eps*k_t
                xN = xS + eps*k_xy[...,0]
                yN = yS + eps*k_xy[...,1]
                muN= muS+ eps*k_mu
                sgN= sgS+ eps*k_sg
            return (jnp.nan_to_num(tN), jnp.nan_to_num(xN), jnp.nan_to_num(yN), jnp.nan_to_num(muN), jnp.nan_to_num(sgN))

        tS,xS,yS,muS,sgS = jax.lax.fori_loop(0, num_steps, body, (tS,xS,yS,muS,sgS))
        return tS,xS,yS,muS,sgS,mW

    def _loss_impl(params_gen, txy_batch):
        txy_batch = jnp.asarray(txy_batch, dtype=jnp.float64)
        tL = txy_batch[:,0]; xL = txy_batch[:,1]; yL = txy_batch[:,2]
        t_push,x_push,y_push,mu_pred,sg_pred,mW = _flow_allgens(params_gen,tL,xL,yL)
        m,B = t_push.shape

        tpc,xpc,ypc = _sanitize(t_push.reshape(-1), x_push.reshape(-1), y_push.reshape(-1))
        mu_eval = _mu_eval(tpc,xpc,ypc).reshape(m,B,2)
        sg_eval,_= _sig_eval(tpc,xpc,ypc)
        sg_eval = sg_eval.reshape(m,B,2,mW)

        mu_mse = jnp.mean((mu_pred-mu_eval)**2, axis=(1,2))
        sg_mse = jnp.mean((sg_pred-sg_eval)**2, axis=(1,2,3))
        dt = t_push - tL[None,:]
        dt_neg = jnp.mean(jax.nn.softplus(-dt), axis=1)
        per_gen = mu_mse + sg_mse + dt_neg_penalty*dt_neg
        loss = jnp.mean(per_gen)
        return loss, {"s7_mu_mse": jnp.mean(mu_mse), "s7_sigma_mse": jnp.mean(sg_mse), "s7_dt_neg": jnp.mean(dt_neg)}

    return jax.jit(_loss_impl) if jit else _loss_impl

# Build loss fns
s1 = make_s1_lie_loss(m)
s2 = make_s2_jacobi_loss_nested(m)
s3 = make_s3_skewsym_loss(m)
s4 = make_s4_bilinearity_loss(m, num_cc=4)
s5 = make_s5_column_independence_loss(m, mode="sigma", tau=0.02)
s6 = make_s6_commutator_loss_ito(mu_fn=mu_fn, sig_fn=sig_fn, use_abs=False)
s7 = make_s7_pushforward_coeff_loss_sde_only_2d(mu_fn=mu_fn, sig_fn=sig_fn, eps=1e-2, num_steps=1, use_heun=True)

# ---------------------------
# 10) Training setup
# ---------------------------
@dataclass
class GenTrainConfig:
    steps: int = 6000
    batch_size: int = 256
    lr: float = 2e-4
    print_every: int = 200
    grad_clip: float = 1.0

gen_train_cfg = GenTrainConfig(**getattr(globals().get("gen_train_cfg", GenTrainConfig()), "__dict__", {}))

def weight_schedule(step, steps):
    s = step / max(1, steps)
    ramp = jnp.clip((s - 0.6) / 0.4, 0.0, 1.0)
    w_s6 = 10.0
    w_s7 = 2.0
    w_s5 = 2.0
    w_s1 = 0.5 * ramp
    w_s2 = 0.2 * ramp
    w_s3 = 0.2 * ramp
    w_s4 = 0.2 * ramp
    return jnp.asarray([w_s1,w_s2,w_s3,w_s4,w_s5,w_s6,w_s7], dtype=jnp.float64)

# Sampler: half empirical (TX_gen) half uniform
TX_gen_np = None
if TX_gen is not None:
    TX_gen_np = np.asarray(jax.device_get(jnp.asarray(TX_gen, dtype=jnp.float64)))
    TX_gen_np = TX_gen_np[np.isfinite(TX_gen_np).all(axis=1)]
    if TX_gen_np.shape[0] == 0:
        TX_gen_np = None

def _infer_bounds_from_TX(TX):
    return float(np.min(TX[:,0])), float(np.max(TX[:,0])), float(np.min(TX[:,1])), float(np.max(TX[:,1])), float(np.min(TX[:,2])), float(np.max(TX[:,2]))

if TX_gen_np is not None:
    tmin_u,tmax_u,xmin_u,xmax_u,ymin_u,ymax_u = _infer_bounds_from_TX(TX_gen_np)
else:
    tmin_u,tmax_u = 0.0, float(getattr(cfg_ex5, "T", 2.0))
    xmin_u,xmax_u = 0.2, 5.0
    ymin_u,ymax_u = -3.0, 5.0

X_FLOOR = max(1e-2, xmin_u)
rng_np = np.random.default_rng(0)

def sample_txy_batch(batch_size):
    n_uni = batch_size//2
    n_emp = batch_size - n_uni
    chunks = []
    if (TX_gen_np is not None) and (n_emp>0):
        N = TX_gen_np.shape[0]
        idx = rng_np.choice(N, size=n_emp, replace=(n_emp>N))
        chunks.append(TX_gen_np[idx])
    if n_uni>0:
        t = rng_np.uniform(tmin_u, tmax_u, size=(n_uni,1))
        x = rng_np.uniform(X_FLOOR, xmax_u, size=(n_uni,1))
        y = rng_np.uniform(ymin_u, ymax_u, size=(n_uni,1))
        chunks.append(np.concatenate([t,x,y], axis=1))
    TXb = np.concatenate(chunks, axis=0)
    rng_np.shuffle(TXb)
    return jnp.asarray(TXb, dtype=jnp.float64)

print("[train] bounds:",
      f"t∈[{tmin_u:.3g},{tmax_u:.3g}]",
      f"x∈[{X_FLOOR:.3g},{xmax_u:.3g}]",
      f"y∈[{ymin_u:.3g},{ymax_u:.3g}]",
      f"| TX_gen = {None if TX_gen_np is None else TX_gen_np.shape[0]}")

# Optimizer
warmup = int(0.05*gen_train_cfg.steps)
cosine = optax.cosine_decay_schedule(init_value=gen_train_cfg.lr, decay_steps=max(1, gen_train_cfg.steps-warmup), alpha=0.1)
schedule = optax.join_schedules([optax.linear_schedule(0.0, gen_train_cfg.lr, warmup), cosine], boundaries=[warmup])

optimizer_gen = optax.chain(
    optax.clip_by_global_norm(gen_train_cfg.grad_clip),
    optax.adamw(learning_rate=schedule, b1=0.9, b2=0.999, eps=1e-8, weight_decay=0.0),
)
opt_state = optimizer_gen.init(params_gen)

@jax.jit
def train_step(params_gen, opt_state, txy_batch, step_idx):
    w = weight_schedule(step_idx, gen_train_cfg.steps)

    L1, aux1 = s1(params_gen, txy_batch)
    L2, aux2 = s2(params_gen, txy_batch)
    L3, aux3 = s3(params_gen, txy_batch)
    L4, aux4 = s4(params_gen, txy_batch, None)
    L5, aux5 = s5(params_gen, txy_batch)
    L6, aux6 = s6(params_gen, txy_batch)
    L7, aux7 = s7(params_gen, txy_batch)

    def loss_fn(p):
        L1_, _ = s1(p, txy_batch)
        L2_, _ = s2(p, txy_batch)
        L3_, _ = s3(p, txy_batch)
        L4_, _ = s4(p, txy_batch, None)
        L5_, _ = s5(p, txy_batch)
        L6_, _ = s6(p, txy_batch)
        L7_, _ = s7(p, txy_batch)
        return w[0]*L1_ + w[1]*L2_ + w[2]*L3_ + w[3]*L4_ + w[4]*L5_ + w[5]*L6_ + w[6]*L7_

    total = loss_fn(params_gen)
    grads = jax.grad(loss_fn)(params_gen)
    updates, opt_state2 = optimizer_gen.update(grads, opt_state, params_gen)
    params2 = optax.apply_updates(params_gen, updates)

    aux = dict(
        total=total, w=w,
        L1=L1, L2=L2, L3=L3, L4=L4, L5=L5, L6=L6, L7=L7,
        sigma_min=aux5.get("sigma_min", jnp.nan),
        s7_mu_mse=aux7.get("s7_mu_mse", jnp.nan),
        s7_sigma_mse=aux7.get("s7_sigma_mse", jnp.nan),
        s7_dt_neg=aux7.get("s7_dt_neg", jnp.nan),
    )
    return params2, opt_state2, aux

# Train
hist = []
for step in range(1, gen_train_cfg.steps+1):
    txy_batch = sample_txy_batch(gen_train_cfg.batch_size)
    params_gen, opt_state, aux = train_step(params_gen, opt_state, txy_batch, step)

    if (step % gen_train_cfg.print_every) == 0 or step == 1 or step == gen_train_cfg.steps:
        w_np = np.asarray(jax.device_get(aux["w"]))
        print(
            f"step {step:5d}/{gen_train_cfg.steps} | total={float(aux['total']):.3e} | "
            f"L1={float(aux['L1']):.2e} L2={float(aux['L2']):.2e} L3={float(aux['L3']):.2e} L4={float(aux['L4']):.2e} | "
            f"L5={float(aux['L5']):.2e} L6={float(aux['L6']):.2e} L7={float(aux['L7']):.2e} | "
            f"sigma_min={float(aux['sigma_min']):.3e} | "
            f"s7(mu,sig,dtneg)=({float(aux['s7_mu_mse']):.2e},{float(aux['s7_sigma_mse']):.2e},{float(aux['s7_dt_neg']):.2e}) | "
            f"w={w_np.round(3)}"
        )

    hist.append(float(aux["total"]))

globals()["params_gen"] = params_gen
globals()["hist_gen_total"] = hist
print("[OK] Training finished. Exported params_gen.")




In [ ]:
# ============================================================
# 11) Robust span check vs GT SDE symmetry span (m=3)
#    v1 = ∂_t
#    v3 = ∂_y
#    v4 = 2t∂_t + x∂_x + (y + a2 t)∂_y
# ============================================================

def _gt_generators_ex5(t, x, y, a2):
    t = jnp.asarray(t, dtype=jnp.float64)
    x = jnp.asarray(x, dtype=jnp.float64)
    y = jnp.asarray(y, dtype=jnp.float64)
    B = t.shape[0]
    v1 = jnp.stack([jnp.ones((B,), dtype=jnp.float64),
                    jnp.zeros((B,), dtype=jnp.float64),
                    jnp.zeros((B,), dtype=jnp.float64)], axis=1)
    v3 = jnp.stack([jnp.zeros((B,), dtype=jnp.float64),
                    jnp.zeros((B,), dtype=jnp.float64),
                    jnp.ones((B,), dtype=jnp.float64)], axis=1)
    v4 = jnp.stack([2.0*t,
                    x,
                    (y + a2*t)], axis=1)
    return jnp.stack([v1, v3, v4], axis=0)  # (3,B,3)

def _stack_columns(V_mB3):
    V = jnp.transpose(V_mB3, (1,2,0))  # (B,3,m)
    return V.reshape(-1, V.shape[-1])  # (3B,m)

def span_metrics(A_gt, A_learn):
    Qg, _ = jnp.linalg.qr(A_gt)
    Ql, _ = jnp.linalg.qr(A_learn)
    M = Qg.T @ Ql
    s = jnp.linalg.svd(M, compute_uv=False)
    s = jnp.clip(s, 0.0, 1.0)
    angles = jnp.arccos(s)

    Pg = Qg @ Qg.T
    Pl = Ql @ Ql.T
    r_learn_to_gt = jnp.linalg.norm((jnp.eye(Pg.shape[0]) - Pg) @ A_learn) / (jnp.linalg.norm(A_learn) + 1e-12)
    r_gt_to_learn = jnp.linalg.norm((jnp.eye(Pl.shape[0]) - Pl) @ A_gt) / (jnp.linalg.norm(A_gt) + 1e-12)

    M_ls, _, _, _ = jnp.linalg.lstsq(A_gt, A_learn, rcond=None)
    fit_rel = jnp.linalg.norm(A_learn - A_gt @ M_ls) / (jnp.linalg.norm(A_learn) + 1e-12)
    return angles, r_learn_to_gt, r_gt_to_learn, fit_rel

# -------- user-editable principal-angle domain (defaults match previous behavior) --------
span_t_min = float(globals().get("span_t_min", tmin_u))
span_t_max = float(globals().get("span_t_max", tmax_u))
span_x_min = float(globals().get("span_x_min", X_FLOOR))
span_x_max = float(globals().get("span_x_max", xmax_u))
span_y_min = float(globals().get("span_y_min", ymin_u))
span_y_max = float(globals().get("span_y_max", ymax_u))

print(
    "\n[SPAN CHECK DOMAIN] "
    f"t∈[{span_t_min:.6g},{span_t_max:.6g}]  "
    f"x∈[{span_x_min:.6g},{span_x_max:.6g}]  "
    f"y∈[{span_y_min:.6g},{span_y_max:.6g}]"
)

Bcheck = 512
t_s = jnp.asarray(rng_np.uniform(span_t_min, span_t_max, size=(Bcheck,)), dtype=jnp.float64)
x_s = jnp.asarray(rng_np.uniform(span_x_min, span_x_max, size=(Bcheck,)), dtype=jnp.float64)
y_s = jnp.asarray(rng_np.uniform(span_y_min, span_y_max, size=(Bcheck,)), dtype=jnp.float64)

tauL, xiL = eval_generators_tau_xi_jit(params_gen, t_s, x_s, y_s, activation=gen_cfg.activation)
V_learn = jnp.stack([tauL, xiL[...,0], xiL[...,1]], axis=-1)  # (m,B,3)
A_learn = _stack_columns(V_learn)                             # (3B,m)

a2 = float(globals().get("truth", {}).get("a2", 0.5))
V_gt = _gt_generators_ex5(t_s, x_s, y_s, a2=a2)               # (3,B,3)
A_gt = _stack_columns(V_gt)                                   # (3B,3)

angles, rL2G, rG2L, fit_rel = span_metrics(A_gt, A_learn)

print("\n===== SPAN CHECK (learned vs GT span{v1,v3,v4}) =====")
print("principal angles (rad):", np.asarray(jax.device_get(angles)))
print("principal angles (deg):", np.asarray(jax.device_get(angles))*180.0/np.pi)
print(f"residual ||(I-P_gt)A_learn||/||A_learn||          = {float(rL2G):.3e}")
print(f"residual ||(I-P_learn)A_gt||/||A_gt||             = {float(rG2L):.3e}")
print(f"best-mixing fit ||A_learn - A_gt M||/||A_learn||  = {float(fit_rel):.3e}")

globals().update({
    "span_angles_rad": angles,
    "span_res_learn_to_gt": rL2G,
    "span_res_gt_to_learn": rG2L,
    "span_bestmix_fit": fit_rel,
})
